## 1. Configuración del Entorno

In [1]:
# Importación de librerías
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

# Configuración
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Importar módulos del proyecto
from utils import (
    calculate_possession,
    calculate_xg,
    calculate_ppda,
    generate_sample_data,
    PUMAS_COLORS
)
from pumas_dashboard_etl import PumasDashboardETL

print("✅ Librerías cargadas exitosamente")
print(f"📦 Pandas version: {pd.__version__}")
print(f"📦 NumPy version: {np.__version__}")

✅ Librerías cargadas exitosamente
📦 Pandas version: 2.3.3
📦 NumPy version: 1.26.4


## 2. Carga y Exploración de Datos

In [2]:
# Inicializar ETL y cargar datos
etl = PumasDashboardETL()

# Cargar datos (usará datos de ejemplo si no existe archivo)
raw_data = etl.load_data()

print(f"\n📊 Datos cargados: {len(raw_data)} partidos")
print(f"📅 Período: {raw_data['date'].min()} a {raw_data['date'].max()}")

2026-01-06 12:55:47,895 - INFO - PumasDashboardETL inicializado
2026-01-06 12:55:47,896 - WARNING - Archivo no encontrado. Generando datos de ejemplo...
2026-01-06 12:55:47,905 - INFO - Datos de ejemplo generados: 17 partidos



📊 Datos cargados: 17 partidos
📅 Período: 2024-01-15 a 2024-05-06


In [3]:
# Vista previa de los datos
print("\n🔍 Vista previa de los datos:")
raw_data.head()


🔍 Vista previa de los datos:


,match_id,date,opponent,is_home,venue,pumas_goals,opponent_goals,result,possession,pumas_passes,opponent_passes,pumas_shots,opponent_shots,pumas_xg,opponent_xg,ppda,build_up_speed,tackles,interceptions,aerial_duels_won,progressive_passes
0,1,2024-01-15,América,True,Estadio Olímpico Universitario,5,0,W,48.55,452,479,15,8,3.35,2.75,3.02,15.75,22,12,22,57
1,2,2024-01-22,Guadalajara,False,Estadio Guadalajara,1,2,L,56.65,460,352,15,14,3.61,2.74,4.70,10.72,25,16,23,76
2,3,2024-01-29,Cruz Azul,True,Estadio Olímpico Universitario,3,2,W,45.92,400,471,15,5,4.85,0.89,3.03,11.87,26,17,18,41
3,4,2024-02-05,Tigres,False,Estadio Tigres,0,5,L,57.94,478,347,12,9,3.36,3.21,3.11,10.81,23,12,17,68
4,5,2024-02-12,Monterrey,True,Estadio Olímpico Universitario,1,3,L,45.14,395,480,10,11,1.73,2.87,3.36,16.23,18,13,22,71


In [4]:
# Información del dataset
print("\n📋 Información del Dataset:")
print(f"   Filas: {raw_data.shape[0]}")
print(f"   Columnas: {raw_data.shape[1]}")
print(f"\n   Columnas disponibles:")
for col in raw_data.columns:
    print(f"   - {col}: {raw_data[col].dtype}")


📋 Información del Dataset:
   Filas: 17
   Columnas: 21

   Columnas disponibles:
   - match_id: int64
   - date: object
   - opponent: object
   - is_home: bool
   - venue: object
   - pumas_goals: int64
   - opponent_goals: int64
   - result: object
   - possession: float64
   - pumas_passes: int64
   - opponent_passes: int64
   - pumas_shots: int64
   - opponent_shots: int64
   - pumas_xg: float64
   - opponent_xg: float64
   - ppda: float64
   - build_up_speed: float64
   - tackles: int64
   - interceptions: int64
   - aerial_duels_won: int64
   - progressive_passes: int64


## 3. Procesamiento de Datos

In [5]:
# Ejecutar transformaciones
df = etl.transform_data()

print("✅ Transformaciones aplicadas")
print(f"   Nuevas columnas creadas: {len(df.columns) - len(raw_data.columns)}")

2026-01-06 12:56:01,387 - INFO - Datos transformados: 17 registros, 36 columnas


✅ Transformaciones aplicadas
   Nuevas columnas creadas: 15


In [6]:
# Calcular métricas agregadas
metrics = etl.calculate_all_metrics()

print("\n📈 Métricas del Equipo:")
print("="*50)
print(f"📊 Record: {metrics['wins']}V - {metrics['draws']}E - {metrics['losses']}D")
print(f"⚽ Goles: {metrics['total_goals_scored']} anotados, {metrics['total_goals_conceded']} recibidos")
print(f"🎯 xG Total: {metrics['total_xg']} (Sobre-rendimiento: {metrics['xg_overperformance_total']})")
print(f"🏃 Posesión Promedio: {metrics['avg_possession']}%")
print(f"💪 PPDA Promedio: {metrics['avg_ppda']}")

2026-01-06 12:56:05,600 - INFO - Métricas calculadas exitosamente



📈 Métricas del Equipo:
📊 Record: 10V - 2E - 5D
⚽ Goles: 55 anotados, 37 recibidos
🎯 xG Total: 64.43 (Sobre-rendimiento: -9.43)
🏃 Posesión Promedio: 51.19%
💪 PPDA Promedio: 2.75


## 4. Análisis de Posesión

In [7]:
# Evolución de la posesión por partido
fig_possession = go.Figure()

# Línea de posesión
fig_possession.add_trace(go.Scatter(
    x=df['match_id'],
    y=df['possession'],
    mode='lines+markers',
    name='Posesión',
    line=dict(color=PUMAS_COLORS['primary'], width=3),
    marker=dict(size=10)
))

# Media móvil
fig_possession.add_trace(go.Scatter(
    x=df['match_id'],
    y=df['possession_rolling'],
    mode='lines',
    name='Media Móvil (5 partidos)',
    line=dict(color=PUMAS_COLORS['secondary'], width=2, dash='dash')
))

# Línea de referencia 50%
fig_possession.add_hline(y=50, line_dash="dot", line_color="gray", 
                         annotation_text="50%")

fig_possession.update_layout(
    title='📊 Evolución de la Posesión por Partido',
    xaxis_title='Partido',
    yaxis_title='Posesión (%)',
    template='plotly_white',
    height=450
)

fig_possession.show()

In [8]:
# Posesión vs Resultado
fig_poss_result = px.box(
    df,
    x='result',
    y='possession',
    color='result',
    color_discrete_map={'W': '#2ecc71', 'D': '#f39c12', 'L': '#e74c3c'},
    category_orders={'result': ['W', 'D', 'L']},
    labels={'result': 'Resultado', 'possession': 'Posesión (%)'},
    title='📊 Distribución de Posesión por Resultado'
)

fig_poss_result.update_layout(template='plotly_white', height=400)
fig_poss_result.show()

## 5. Análisis de Expected Goals (xG)

In [9]:
# xG vs Goles reales
fig_xg = go.Figure()

# Goles reales
fig_xg.add_trace(go.Bar(
    x=df['opponent'],
    y=df['pumas_goals'],
    name='Goles Reales',
    marker_color=PUMAS_COLORS['primary']
))

# xG
fig_xg.add_trace(go.Scatter(
    x=df['opponent'],
    y=df['pumas_xg'],
    mode='markers+lines',
    name='Expected Goals (xG)',
    marker=dict(size=12, color=PUMAS_COLORS['secondary']),
    line=dict(color=PUMAS_COLORS['secondary'], width=2)
))

fig_xg.update_layout(
    title='🎯 Goles Reales vs Expected Goals (xG) por Rival',
    xaxis_title='Rival',
    yaxis_title='Goles / xG',
    template='plotly_white',
    barmode='group',
    height=450
)

fig_xg.show()

In [10]:
# Scatter plot: xG vs xGA con resultado
fig_xg_scatter = px.scatter(
    df,
    x='pumas_xg',
    y='opponent_xg',
    color='result',
    size='possession',
    hover_data=['opponent', 'pumas_goals', 'opponent_goals'],
    color_discrete_map={'W': '#2ecc71', 'D': '#f39c12', 'L': '#e74c3c'},
    labels={
        'pumas_xg': 'xG Pumas',
        'opponent_xg': 'xG Rival',
        'result': 'Resultado'
    },
    title='🎯 xG Pumas vs xG Rival (tamaño = posesión)'
)

# Línea de referencia diagonal
fig_xg_scatter.add_trace(go.Scatter(
    x=[0, df['pumas_xg'].max()],
    y=[0, df['pumas_xg'].max()],
    mode='lines',
    name='Línea de equilibrio',
    line=dict(color='gray', dash='dash')
))

fig_xg_scatter.update_layout(template='plotly_white', height=500)
fig_xg_scatter.show()

## 6. Análisis de Pressing (PPDA)

In [11]:
# PPDA por partido
fig_ppda = go.Figure()

# Colorear barras según intensidad
colors = ['#2ecc71' if x < 8 else '#f39c12' if x < 12 else '#e74c3c' 
          for x in df['ppda']]

fig_ppda.add_trace(go.Bar(
    x=df['opponent'],
    y=df['ppda'],
    marker_color=colors,
    text=df['ppda'].round(1),
    textposition='outside'
))

# Líneas de referencia
fig_ppda.add_hline(y=8, line_dash="dot", line_color="green",
                   annotation_text="Pressing Alto")
fig_ppda.add_hline(y=12, line_dash="dot", line_color="red",
                   annotation_text="Pressing Bajo")

fig_ppda.update_layout(
    title='💪 PPDA (Passes Allowed Per Defensive Action) por Rival',
    xaxis_title='Rival',
    yaxis_title='PPDA (menor = mejor pressing)',
    template='plotly_white',
    height=450
)

fig_ppda.show()

In [12]:
# Distribución de intensidad de pressing
pressing_counts = df['pressing_intensity'].value_counts()

fig_pressing_pie = px.pie(
    values=pressing_counts.values,
    names=pressing_counts.index,
    color=pressing_counts.index,
    color_discrete_map={'Alto': '#2ecc71', 'Medio': '#f39c12', 'Bajo': '#e74c3c'},
    title='🥧 Distribución de Intensidad de Pressing'
)

fig_pressing_pie.update_traces(textposition='inside', textinfo='percent+label')
fig_pressing_pie.update_layout(height=400)
fig_pressing_pie.show()

## 7. Velocidad de Construcción

In [14]:
# Velocidad de construcción vs Posesión
fig_build = px.scatter(
    df,
    x='possession',
    y='build_up_speed',
    color='result',
    size='pumas_xg',
    hover_data=['opponent'],
    color_discrete_map={'W': '#2ecc71', 'D': '#f39c12', 'L': '#e74c3c'},
    labels={
        'possession': 'Posesión (%)',
        'build_up_speed': 'Velocidad de Construcción (s)',
        'result': 'Resultado'
    },
    title='⏱️ Velocidad de Construcción vs Posesión'
)

fig_build.update_layout(template='plotly_white', height=450)
fig_build.show()

## 8. Dashboard Resumen

In [15]:
# Dashboard con múltiples métricas
fig_dashboard = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Puntos Acumulados',
        'xG Acumulado',
        'Posesión (Media Móvil)',
        'PPDA (Media Móvil)'
    ),
    specs=[[{"type": "scatter"}, {"type": "scatter"}],
           [{"type": "scatter"}, {"type": "scatter"}]]
)

# Puntos acumulados
fig_dashboard.add_trace(
    go.Scatter(x=df['match_id'], y=df['cumulative_points'],
               mode='lines+markers', name='Puntos',
               line=dict(color=PUMAS_COLORS['primary'], width=3)),
    row=1, col=1
)

# xG acumulado
fig_dashboard.add_trace(
    go.Scatter(x=df['match_id'], y=df['cumulative_xg'],
               mode='lines+markers', name='xG Acum.',
               line=dict(color=PUMAS_COLORS['secondary'], width=3)),
    row=1, col=2
)
fig_dashboard.add_trace(
    go.Scatter(x=df['match_id'], y=df['cumulative_goals'],
               mode='lines', name='Goles Acum.',
               line=dict(color='#2ecc71', width=2, dash='dash')),
    row=1, col=2
)

# Posesión rolling
fig_dashboard.add_trace(
    go.Scatter(x=df['match_id'], y=df['possession_rolling'],
               mode='lines+markers', name='Posesión MM',
               fill='tozeroy',
               line=dict(color=PUMAS_COLORS['primary'])),
    row=2, col=1
)

# PPDA rolling
fig_dashboard.add_trace(
    go.Scatter(x=df['match_id'], y=df['ppda_rolling'],
               mode='lines+markers', name='PPDA MM',
               fill='tozeroy',
               line=dict(color='#e74c3c')),
    row=2, col=2
)

fig_dashboard.update_layout(
    height=700,
    showlegend=False,
    title_text='📊 Dashboard de Métricas - Pumas UNAM',
    template='plotly_white'
)

fig_dashboard.show()

## 9. Análisis Local vs Visitante

In [16]:
# Comparativa local vs visitante
home_away_stats = df.groupby('is_home').agg({
    'points': 'sum',
    'pumas_goals': 'sum',
    'opponent_goals': 'sum',
    'possession': 'mean',
    'pumas_xg': 'mean',
    'ppda': 'mean'
}).round(2)

home_away_stats.index = ['Visitante', 'Local']
home_away_stats.columns = ['Puntos', 'Goles', 'Goles Recibidos', 'Posesión', 'xG Prom.', 'PPDA']

print("\n🏟️ Comparativa Local vs Visitante:")
home_away_stats


🏟️ Comparativa Local vs Visitante:


,Puntos,Goles,Goles Recibidos,Posesión,xG Prom.,PPDA
Visitante,14,24,21,52.42,3.63,2.73
Local,18,31,16,50.09,3.93,2.77


In [17]:
# Visualización radar local vs visitante
categories = ['Puntos/Partido', 'xG', 'Posesión', 'Inv. PPDA']

home_stats = df[df['is_home']]
away_stats = df[~df['is_home']]

# Normalizar valores para radar
home_values = [
    home_stats['points'].mean() / 3 * 100,
    home_stats['pumas_xg'].mean() / 3 * 100,
    home_stats['possession'].mean(),
    (15 - home_stats['ppda'].mean()) / 15 * 100  # Invertir PPDA
]

away_values = [
    away_stats['points'].mean() / 3 * 100,
    away_stats['pumas_xg'].mean() / 3 * 100,
    away_stats['possession'].mean(),
    (15 - away_stats['ppda'].mean()) / 15 * 100
]

fig_radar = go.Figure()

fig_radar.add_trace(go.Scatterpolar(
    r=home_values + [home_values[0]],
    theta=categories + [categories[0]],
    fill='toself',
    name='Local',
    line_color=PUMAS_COLORS['primary']
))

fig_radar.add_trace(go.Scatterpolar(
    r=away_values + [away_values[0]],
    theta=categories + [categories[0]],
    fill='toself',
    name='Visitante',
    line_color=PUMAS_COLORS['secondary']
))

fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    showlegend=True,
    title='🎯 Perfil de Rendimiento: Local vs Visitante',
    height=500
)

fig_radar.show()

## 10. Exportación para Power BI / Tableau

In [18]:
# Exportar todos los datos procesados
exports = etl.export_all()

print("\n📁 Archivos exportados para BI:")
for name, path in exports.items():
    print(f"   ✅ {name}: {path}")

2026-01-06 12:57:20,905 - INFO - Archivo exportado exitosamente: output\pumas_processed_data.csv
2026-01-06 12:57:20,907 - INFO - Datos exportados para Power BI: output\pumas_processed_data.csv
2026-01-06 12:57:20,916 - INFO - Archivo exportado exitosamente: output\pumas_metrics_summary.csv
2026-01-06 12:57:20,928 - INFO - Archivo exportado exitosamente: output\pumas_match_timeline.csv
2026-01-06 12:57:20,930 - INFO - Exportación completa: 4 archivos



📁 Archivos exportados para BI:
   ✅ processed_data: output\pumas_processed_data.csv
   ✅ metrics_summary: output\pumas_metrics_summary.csv
   ✅ match_timeline: output\pumas_match_timeline.csv
   ✅ metrics_json: output\pumas_metrics.json


In [19]:
# Resumen de métricas para KPIs del dashboard
summary = etl.create_metrics_summary()
print("\n📊 Resumen de KPIs para Dashboard:")
summary


📊 Resumen de KPIs para Dashboard:


,metric,value
0,total_matches,17.00
1,wins,10.00
2,draws,2.00
3,losses,5.00
4,total_points,32.00
5,points_per_match,1.88
6,total_goals_scored,55.00
7,total_goals_conceded,37.00
8,goal_difference,18.00
9,goals_per_match,3.24


## 11. Conclusiones y Siguiente Pasos

### 📈 Insights Principales

1. **Posesión:** El equipo mantiene un promedio de posesión que refleja su estilo de juego
2. **xG:** La relación entre xG y goles reales indica la eficiencia del ataque
3. **Pressing:** El PPDA muestra la intensidad defensiva del equipo
4. **Local vs Visitante:** Diferencias significativas en rendimiento según la localía

### 🔜 Siguientes Pasos

1. Importar los CSV generados en Power BI / Tableau
2. Crear visualizaciones interactivas con filtros por:
   - Rival
   - Fecha
   - Localía
   - Resultado
3. Agregar datos de jugadores individuales
4. Implementar actualización automática de datos

---

**Autor:** César Adrián Delgado Díaz  
**GitHub:** [github.com/cesar530](https://github.com/cesar530)